# 實作本文分類
### 程式來自[Text classification with the torchtext library](https://pytorch.org/tutorials/beginner/text_sentiment_ngrams_tutorial.html)

## 載入AG News資料集 

In [1]:
from __future__ import annotations

import time
from collections import Counter, OrderedDict
from typing import Callable, Iterable, Iterator, cast, overload

import torch
from datasets import Dataset as HFDataset
from datasets import load_dataset
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, random_split
from text_utils import get_tokenizer
from text_utils import vocab as build_vocab

In [2]:
# torchtext.datasets.AG_NEWS 已棄用，改用 Hugging Face datasets 載入 AG News 資料集
# 標籤對照：HF 資料集標籤為 0~3，torchtext 版本為 1~4，故一律 +1 以維持相容
NewsIterator = Iterator[tuple[int, str]]


@overload
def AG_NEWS(split: str) -> NewsIterator: ...
@overload
def AG_NEWS(split: tuple[str, str] = ('train', 'test')) -> tuple[NewsIterator, NewsIterator]: ...
def AG_NEWS(
    split: str | tuple[str, str] = ('train', 'test'),
) -> NewsIterator | tuple[NewsIterator, NewsIterator]:
    single = isinstance(split, str)
    splits = (split,) if single else split
    results = []
    for s in splits:
        hf_split = 'test' if s == 'test' else 'train'
        ds = cast(HFDataset, load_dataset('fancyzhx/ag_news', split=hf_split))
        results.append((cast(dict, example)['label'] + 1, cast(dict, example)['text']) for example in ds)
    return results[0] if single else tuple(results)


news = AG_NEWS(split='train')

type(news)

generator

In [3]:
train_iter = iter(AG_NEWS(split='train'))

In [4]:
# 取得下一筆資料
next(train_iter)

(3,
 "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")

## 判斷GPU是否存在

In [5]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
device

'mps'

## 詞彙表處理

In [6]:
# 分詞
tokenizer = get_tokenizer('basic_english')


# 建立 Generator 函數
def yield_tokens(data_iter: Iterable[tuple[int, str]]) -> Iterator[list[str]]:
    for _, text in data_iter:
        yield tokenizer(text)


# 由 train_iter 建立詞彙字典(統計詞頻，再依出現次數降冪排列)
counter = Counter()
for tokens in yield_tokens(train_iter):
    counter.update(tokens)
sorted_by_freq_tuples = sorted(counter.items(), key=lambda x: x[1], reverse=True)
ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = build_vocab(ordered_dict, specials=["<unk>"])

# 設定預設的索引值
vocab.set_default_index(vocab["<unk>"])

In [7]:
# 測試詞彙字典，取得單字的索引值
vocab.lookup_indices(['here', 'is', 'an', 'example'])

[475, 21, 30, 5286]

## 參數設定

In [8]:
EPOCHS = 10  # 訓練週期數
LR = 5  # 學習率
BATCH_SIZE = 64  # 訓練批量
# 取得標註個數
num_class = len(set([label for (label, text) in news]))
vocab_size = len(vocab)
emsize = 64

## 定義資料轉換函數

In [9]:
text_pipeline = lambda x: vocab.lookup_indices(tokenizer(x))  # 分詞、取得單字的索引值
label_pipeline = lambda x: int(x) - 1  # 換成索引值

In [10]:
# 測試資料轉換
print(text_pipeline('here is an example'))
label_pipeline('10')

[475, 21, 30, 5286]


9

## 建立模型

In [11]:
class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_class: int) -> None:
        super().__init__()
        # sparse=True 會導致新版 PyTorch 的 clip_grad_norm_ 在 CPU 上失敗(不支援稀疏梯度)，故改為 False
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        self.fc = nn.Linear(embed_dim, num_class)
        self.init_weights()

    def init_weights(self) -> None:
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text: torch.Tensor, offsets: torch.Tensor) -> torch.Tensor:
        embedded: torch.Tensor = self.embedding(text, offsets)
        return self.fc(embedded)


model = TextClassificationModel(vocab_size, emsize, num_class).to(device)

## 定義訓練及評估函數

In [12]:
# 訓練函數
def train(dataloader: DataLoader) -> None:
    model.train()
    total_acc, total_count = 0.0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (label, text, offsets) in enumerate(dataloader):
        optimizer.zero_grad()
        predicted_label: torch.Tensor = model(text, offsets)
        loss: torch.Tensor = criterion(predicted_label, label)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print(
                f'| epoch {epoch:3d} | {idx:5d}/{len(dataloader):5d} batches '
                f'| accuracy {total_acc / total_count:8.3f}'
            )
            total_acc, total_count = 0.0, 0
            start_time = time.time()


# 評估函數
def evaluate(dataloader: DataLoader) -> float:
    model.eval()
    total_acc, total_count = 0.0, 0

    with torch.no_grad():
        for idx, (label, text, offsets) in enumerate(dataloader):
            predicted_label: torch.Tensor = model(text, offsets)
            loss: torch.Tensor = criterion(predicted_label, label)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc / total_count

## 建立DataLoader，逐批訓練

In [13]:
# torchtext.data.functional.to_map_style_dataset 已棄用，改用 list 即可取得支援 __len__/__getitem__ 的資料集
def to_map_style_dataset(data_iter: Iterable[tuple[int, str]]) -> list[tuple[int, str]]:
    return list(data_iter)


# 批次處理
def collate_batch(batch: Iterable[tuple[str, str]]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    label_list, text_list, offsets = [], [], [0]
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))  # 設定每筆資料的起始位置
    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)  # 每筆資料的起始位置累加
    text_list = torch.cat(text_list)
    return label_list.to(device), text_list.to(device), offsets.to(device)


train_iter, test_iter = AG_NEWS()
# 轉換為 DataSet
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)
# 資料切割，95% 作為訓練資料
num_train = int(len(train_dataset) * 0.95)
split_train_, split_valid_ = random_split(
    cast(Dataset[tuple[int, str]], train_dataset), [num_train, len(train_dataset) - num_train]
)

# 建立DataLoader
train_dataloader = DataLoader(split_train_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(split_valid_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(
    cast(Dataset[tuple[int, str]], test_dataset), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)

## 模型訓練

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.1)

total_accu = None
for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    train(train_dataloader)
    accu_val = evaluate(valid_dataloader)
    if total_accu is not None and total_accu > accu_val:
        scheduler.step()
    else:
        total_accu = accu_val
    print('-' * 59)
    print(
        f'| end of epoch {epoch:3d} | time: {time.time() - epoch_start_time:5.2f}s | '
        f'valid accuracy {accu_val:8.3f} '
    )
    print('-' * 59)

| epoch   1 |   500/ 1782 batches | accuracy    0.684
| epoch   1 |  1000/ 1782 batches | accuracy    0.856
| epoch   1 |  1500/ 1782 batches | accuracy    0.875
-----------------------------------------------------------
| end of epoch   1 | time: 32.33s | valid accuracy    0.891 
-----------------------------------------------------------
| epoch   2 |   500/ 1782 batches | accuracy    0.899
| epoch   2 |  1000/ 1782 batches | accuracy    0.901
| epoch   2 |  1500/ 1782 batches | accuracy    0.902
-----------------------------------------------------------
| end of epoch   2 | time: 33.01s | valid accuracy    0.903 
-----------------------------------------------------------
| epoch   3 |   500/ 1782 batches | accuracy    0.913
| epoch   3 |  1000/ 1782 batches | accuracy    0.915
| epoch   3 |  1500/ 1782 batches | accuracy    0.914
-----------------------------------------------------------
| end of epoch   3 | time: 30.47s | valid accuracy    0.909 
-------------------------------

## 模型評估

In [15]:
print(f'測試資料準確度: {evaluate(test_dataloader):.3f}')

測試資料準確度: 0.909


## 測試新資料

In [16]:
# 新聞類別
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tec"}


# 預測
def predict(text: str, text_pipeline: Callable[[str], list[int]]) -> int:
    with torch.no_grad():
        text_tensor = torch.tensor(text_pipeline(text)).to(device)
        output = model(text_tensor, torch.tensor([0]).to(device))
        return output.argmax(1).item() + 1


# 測試資料
ex_text_str = "MEMPHIS, Tenn. – Four days ago, Jon Rahm was \
    enduring the season’s worst weather conditions on Sunday at The \
    Open on his way to a closing 75 at Royal Portrush, which \
    considering the wind and the rain was a respectable showing. \
    Thursday’s first round at the WGC-FedEx St. Jude Invitational \
    was another story. With temperatures in the mid-80s and hardly any \
    wind, the Spaniard was 13 strokes better in a flawless round. \
    Thanks to his best putting performance on the PGA Tour, Rahm \
    finished with an 8-under 62 for a three-stroke lead, which \
    was even more impressive considering he’d never played the \
    front nine at TPC Southwind."

print(ag_news_label[predict(ex_text_str, text_pipeline)])

Sports


In [17]:
my_test = open('nlp_data/news.txt', encoding='utf8').read()
print(ag_news_label[predict(my_test, text_pipeline)])

Business
